# Spark MLlib – Preprocesamiento de Datos

**Curso:** Computación de Alto Desempeño – MLlib Spark  
**Autor:** Santiago Gil Gallego (Sgg)  
**Fecha:** 27 de noviembre de 2025  

**Objetivo del cuaderno:**  
Este cuaderno realiza la clasificación binaria usando el dataset de cáncer de mama preprocesado y se calcula su matriz de confusión.


In [1]:
import findspark
findspark.init()

from pyspark import SparkConf
from pyspark.sql import SparkSession, SQLContext

configuraSgg = (
    SparkConf()
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.executor.cores", "1")
    .set("spark.executor.memory", "4G")
    .set("spark.cores.max", "2")
    .setMaster("spark://10.43.100.121:7077")
)
configuraSgg.setAppName("hpcsparkSgg_supervisado_cluster")

sparkSgg = SparkSession.builder.config(conf=configuraSgg).getOrCreate()
sqlContext = SQLContext(sparkContext=sparkSgg.sparkContext,
                        sparkSession=sparkSgg)

print("MASTER ACTUAL:", sparkSgg.sparkContext.master)

df_sup = sparkSgg.read.parquet("data/supervised/preprocessed_sgg")
df_sup.printSchema()
df_sup.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 16:42:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


MASTER ACTUAL: spark://10.43.100.121:7077


root
 |-- label: integer (nullable = true)
 |-- features: vector (nullable = true)



+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|[1.09609952943171...|
|    1|[-0.6206757760066...|
|    1|[-0.4731182290929...|
|    0|[0.48884347097913...|
|    0|[1.28338410820681...|
+-----+--------------------+
only showing top 5 rows



In [2]:
train_df, test_df = df_sup.randomSplit([0.7, 0.3], seed=42)
print("Train rows:", train_df.count(), "Test rows:", test_df.count())


Train rows: 426 Test rows: 143


In [3]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

label_col = "label"

lr = LogisticRegression(featuresCol="features", labelCol=label_col)
pipeline = Pipeline(stages=[lr])

model = pipeline.fit(train_df)
predictions = model.transform(test_df)

predictions.select(label_col, "prediction", "probability").show(10, truncate=False)

# Accuracy
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)

# F1
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)
f1 = evaluator_f1.evaluate(predictions)

# AUC ROC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC ROC: {auc:.4f}")


25/11/26 16:43:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |1.0       |[0.0,1.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
+-----+----------+-----------+
only showing top 10 rows



Accuracy: 0.9720
F1-score: 0.9720
AUC ROC: 0.9960


In [4]:
confusion = (
    predictions
    .groupBy(label_col, "prediction")
    .count()
    .orderBy(label_col, "prediction")
)
confusion.show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|   56|
|    0|       1.0|    3|
|    1|       0.0|    1|
|    1|       1.0|   83|
+-----+----------+-----+



## Conclusiones

- Se entrenó un modelo de regresión logística para clasificar tumores como benignos o malignos.
- El modelo obtuvo aproximadamente:
  - **Accuracy:** `0.9720`
  - **F1-score:** `0.9720`
  - **AUC ROC:** `0.9960`
- Estos resultados indican que el modelo tiene un buen desempeño para este dataset clásico y sirven como línea base.
- Como trabajo futuro se podrían:
  - Probar modelos más complejos (árboles, Random Forest).
  - Realizar ajuste de hiperparámetros.
  - Analizar el impacto de la selección de características.
